### Set up and start interactive session


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import regexp_replace, col

args = getResolvedOptions(sys.argv, ["JOB_NAME", "year", "month"])
year = args["year"]
month = args["month"]

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args["JOB_NAME"], args)

In [ ]:
year = "2025"
raw_path = f"s3://chicago-taxi-trips-guanyiw/raw/year={year}/month={month}/taxi_{year}_{month}.csv"

### Schema Enforcement

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    DoubleType, TimestampType
)

schema = StructType([
    StructField("Trip ID", StringType(), True),
    StructField("Taxi ID", StringType(), True),
    StructField("Trip Start Timestamp", TimestampType(), True),
    StructField("Trip End Timestamp", TimestampType(), True),

    StructField("Trip Seconds", LongType(), True),
    StructField("Trip Miles", DoubleType(), True),
    
    StructField("Pickup Census Tract", StringType(), True),
    StructField("Dropoff Census Tract", StringType(), True),

    StructField("Pickup Community Area", IntegerType(), True),
    StructField("Dropoff Community Area", IntegerType(), True),

    StructField("Fare", StringType(), True),
    StructField("Tips", StringType(), True),
    StructField("Tolls", StringType(), True),
    StructField("Extras", StringType(), True),
    StructField("Trip Total", StringType(), True),

    StructField("Payment Type", StringType(), True),
    StructField("Company", StringType(), True),

    StructField("Pickup Centroid Latitude", DoubleType(), True),
    StructField("Pickup Centroid Longitude", DoubleType(), True),
    StructField("Pickup Centroid Location", DoubleType(), True),
    StructField("Dropoff Centroid Latitude", DoubleType(), True),
    StructField("Dropoff Centroid Longitude", DoubleType(), True),
    StructField("Dropoff Centroid Location", DoubleType(), True)
])

In [ ]:
df = (spark.read
      .option("header", "true")
      .option("mode", "PERMISSIVE")
      .option("timestampFormat", "MM/dd/yyyy hh:mm:ss a")
      .schema(schema)
      .csv(raw_path)
)

### Data Profiling

In [ ]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F

# df = df.dropDuplicates(["Trip ID"])

df = df.filter(col("Trip Start Timestamp").isNotNull()) \
       .filter(col("Trip End Timestamp").isNotNull()) \
       .filter(col("Trip Seconds").isNotNull()) \
       .filter(col("Trip Miles").isNotNull())

df = df.filter(col("Trip Seconds") >= 0)
df = df.filter(col("Trip Miles") >= 0)

df = df.filter(col("Trip End Timestamp") >= col("Trip Start Timestamp"))

df = df.withColumn("year", F.year(col("Trip Start Timestamp"))) \
       .withColumn("month", F.month(col("Trip Start Timestamp"))) \
       .withColumn("pickup_hour", F.hour(col("Trip Start Timestamp"))) \
       .withColumn("pickup_dow", F.dayofweek(col("Trip Start Timestamp")))

money_cols = ["Fare","Tips","Tolls","Extras","Trip Total"]
for c in money_cols:
    df = df.withColumn(c, regexp_replace(col(c), "\\$", "").cast("double"))

df = df.drop("Pickup Census Tract", "Dropoff Census Tract", "Pickup Centroid Location", "Dropoff Centroid Location")

### Renaming

In [ ]:
rename_map = {
    "Trip ID": "trip_id",
    "Taxi ID": "taxi_id",
    "Trip Start Timestamp": "trip_start_timestamp",
    "Trip End Timestamp": "trip_end_timestamp",
    "Trip Seconds": "trip_seconds",
    "Trip Miles": "trip_miles",
    "Pickup Community Area": "pickup_community_area",
    "Dropoff Community Area": "dropoff_community_area",
    "Fare": "fare",
    "Tips": "tips",
    "Tolls": "tolls",
    "Extras": "extras",
    "Trip Total": "trip_total",
    "Payment Type": "payment_type",
    "Company": "company",
    "Pickup Centroid Latitude": "pickup_centroid_latitude",
    "Pickup Centroid Longitude": "pickup_centroid_longitude",
    "Dropoff Centroid Latitude": "dropoff_centroid_latitude",
    "Dropoff Centroid Longitude": "dropoff_centroid_longitude",
}

for old, new in rename_map.items():
    df = df.withColumnRenamed(old, new)

In [ ]:
# print("Total rows:", df.count())
# df.printSchema()

### Write into parquet files

In [ ]:
silver_path = f"s3://chicago-taxi-trips-guanyiw/silver/year={year}/month={month}/"

df = df.repartition(14, "year", "month")
df.write \
  .mode("overwrite") \
  .partitionBy("year", "month") \
  .parquet(silver_path)

In [ ]:
# check = spark.read.parquet("s3://chicago-taxi-trips-guanyiw/silver/year=2025/month=9/")
# print(len(check.columns))
# check.printSchema()